# Blackboard

https://arxiv.org/pdf/2507.01701

In [3]:
question = 'Целеполагание для развития г. Гатчина (Ленинградская область)'

In [ ]:
from pydantic import BaseModel, Field

class BaseMessage(BaseModel):
    content : str = Field(description='Содержание сообщения')

class Message(BaseMessage):
    author : str = Field(description='Автор сообщения')

class BlackBoard(BaseModel):
    question : str
    result : str | None = Field(default=None)

    public_messages : list[Message] = Field(default=[])

    controller_agent = ...
    
    def run(self, max_iter : int = 10):
        
        for i in range(1, max_iter+1):
            ...

## Agents

In [ ]:
from src import llm
from langchain.messages import SystemMessage, HumanMessage, AIMessage

system_prompt = ''

### Controller

In [ ]:
class ControllerResponse(BaseModel):
    agents : list[str] = Field(description='Упорядоченный список ID агентов')

controller_prompt = """
Твоя задача заключается в составлении порядка вызова других агентов, которые будут сотрудничать для решения заданной задачи.

Your task is to schedule other agents to cooperate and solve the
given problem. The agent names and descriptions are listed below:\n
{role_list}. The given problem is:{question}. Agents are sharing
information on the blackboard. Based on the contents existed on the
blackboard, you need to choose suitable agents from agent list to
write on the blackboard.
"""



### Experts

In [ ]:
class Role

class PredefinedRole()

class ExpertRole(BaseModel):
    name : str = Field(max_length=140, description='Название экспертной роли')
    description : str = Field(max_length=140, description='Описание экспертной роли')

class GeneratorResponse(BaseModel):
    roles : list[ExpertRole] = Field(min_length=1, max_length=10, description="Список экспертных ролей")

generator_prompt = """
Тебе задан вопрос. Составь список из экспертных ролей, которые наиболее полезны для решения этого вопроса. 
Вопрос: {question}
"""

roles = llm.with_structured_output(GeneratorResponse).invoke([
    SystemMessage(generator_prompt.format(question=question))
]).roles

roles

[Role(name='Градостроитель‑проектировщик', description='Разрабатывает генеральные планы, схемы застройки, определяет зоны развития, плотность населения и инфраструктурные требования.'),
 Role(name='Экономист‑региональный аналитик', description='Оценивает экономический потенциал Гатчины, формирует цели по привлечению инвестиций, развитию промышленности, туризма и малого бизнеса.'),
 Role(name='Транспортный инженер', description='Планирует развитие дорожной сети, общественного транспорта, велосипедных и пешеходных маршрутов, интеграцию с Санкт‑Петербургом.'),
 Role(name='Экологический специалист', description='Оценивает влияние планируемых проектов на природные ресурсы, формирует цели по сохранению зелёных зон, водных объектов и снижению загрязнён\u200b'),
 Role(name='Социолог‑специалист по общественному развитию', description='Исследует потребности жителей, демографические тенденции, уровень качества жизни; формирует цели в сфере образования, здравоохранения, жилья'),
 Role(name='Экспер

### Decider agent

In [ ]:



manager_tools = [tool for tool in tools if tool.name not in ['add_result']]

manager_prompt = """
Ты менеджер. 

ТВОИ ОБЯЗАННОСТИ:
- Декомпозиция запроса пользователя и создание задач для агентов.
- Отслеживание прогресса по задачам.
- Суммаризация результатов и формирование ответа пользователю.

ТЫ ТАКЖЕ МОЖЕШЬ:
- Помечать задачи как DONE или CANCELLED.
- Менять приоритет задач. 

ВАЖНО:
- Работа заканчивается тогда, когда все задачи на доске либо CANCELLED, либо DONE.
- Если ты готов отдать ответ пользователю, убедись, что все задачи имеют необходимый статус для завершения.
- Агенты ничего не знают про запрос пользователя. При постановке задачи убедись, что информации достаточно.

ТЫ НЕ ДОЛЖЕН отвечать на основе своих знаний. Отвечай только на основе результатов по задачам.

ЗАПРОС ПОЛЬЗОВАТЕЛЯ:
{query}
"""

manager_agent = Agent(tools=manager_tools, system_prompt=manager_prompt.format(query="""
Анализ потенциала г. Гатчина (Ленинградская область)
"""))

def manager_node(state : State):
    board.update_iteration()
    
    messages = state['messages']
    last_messages_len = state.get('last_messages_len',0)
    new_messages = messages[last_messages_len:]

    result = manager_agent.run(new_messages)

    return {
        'manager_response': result,
        'last_messages_len': len(messages)  # обновляем индекс
    }

In [10]:
system_prompt = """
You are {role_name} cooperating with other agents to solve the problem. 
The problem is: {question}.
There is a blackboard that everyone of you can read or write messages.
"""

decider = """
If you think the messages on the blackboard enough to get
the final answer then You should output the final answer with your
answer in the form {{the final answer is boxed[answer]}}, at the
end of your response. otherwise you need other agents provide more
information then say {{\"continue, waiting for more information\"}}
and wait other agent giving new factors. do not output other
information.
"""

system_prompts['planner']

'\nGenerate plans to solve the original problem based on blackboard contents. Strictly follow the json format as follows: {{"\n        [problem]":string //describe the problem,"[planning]":string //was\n        the solving plan of the problem}}, If there already have plan or\n        problem is simple enough to solve then say \\{{"there is no need to\n        decompose tasks, waiting for more information"}}\\. Do not solve\n        the task.\n    '